# 📬 Email Triage + Extraction — demo en Google Colab

Sistema que clasifica correos de solicitudes de un equipo de finanzas/datos, asigna urgencia y extrae los campos clave (solicitante, fecha límite, montos, IDs, tareas) usando modelos de lenguaje preentrenados pequeños, sin entrenamiento.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`. En CPU también funciona, pero la evaluación tarda más.

## 1. Clonar el repositorio e instalar dependencias

In [ ]:
# Repositorio del proyecto
!git clone https://github.com/mlau1818/email-triage-llm.git
%cd email-triage-llm
!pip install -q -r requirements.txt

## 2. (Opcional) Regenerar los datasets
Los CSV ya vienen en `data/`. Estos comandos los vuelven a crear de forma idéntica (semilla fija).

In [ ]:
!python src/generate_data.py
!python src/make_handwritten.py

## 3. Probar un correo desde Python

In [ ]:
import sys; sys.path.insert(0, "src")
from triage import EmailTriage

triage = EmailTriage()   # descarga bart-large-mnli y flan-t5-base la primera vez (~2 GB)

email = """From: Rita Gomez <rita.gomez@example.com>
Subject: numbers don't tie

The dashboard has CC-3301 at $48K but SAP says $41K. Something is off, can someone take a look by Friday?

Thanks,
Rita Gomez"""

triage.triage(email)

## 4. Evaluación en los dos datasets
Genera `results/metrics.json`, las predicciones por correo y las matrices de confusión.

In [ ]:
!python src/evaluate.py

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
print(json.dumps(json.load(open("results/metrics.json")), indent=2))
display(Image("results/confusion_emails.png", width=550))
display(Image("results/confusion_emails_handwritten.png", width=550))
pd.read_csv("results/predictions_emails_handwritten.csv")[
    ["email_id", "category", "pred_category", "urgency", "pred_urgency", "pred_summary", "pred_tasks"]]

## 5. Interfaz gráfica (Gradio)
Abre el enlace público `https://....gradio.live` que aparece abajo. Para detenerla, para la celda.

In [ ]:
!python src/app.py --share